# Extração de Dados SRAG

Este notebook documenta a **ingestão** do Parquet processado do SRAG 2019–2026.

Fluxo:

```
GitHub Release (dados-v2/srag.parquet)
        ↓
get_parquet()  → Path
        ↓
pd.read_parquet()  → DataFrame
        ↓
verificações de sanidade
```

> **Escopo:** aqui só se faz **download e verificação** do artefato canônico.
> Modelagem, validação e métricas ficam em notebooks posteriores.
>
> O download da fonte **bruta** do OpenDataSUS (que produz
> `data/raw/INFLUD*.parquet`) é uma via separada, para auditoria e
> reingestão, e não é o caminho consumido por este notebook.

## 1. Importações e configuração inicial

Funciona tanto se o notebook for executado da **raiz do projeto** quanto de dentro de `notebooks/`.

Aproveita `src.config` para:
- carregar o `.env` (se existir);
- garantir que `data/raw`, `data/processed`, `data/treino` e `reports/figures` existam.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


# Raiz do projeto (funciona tanto da raiz quanto de notebooks/)
ROOT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

# Permite importar src
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


# Módulos internos
from src import config as env
from src import config as C
from src import dados as storage
from src.dados import get_parquet


# Carrega .env (se existir) e garante diretórios padrão do projeto
env.carregar_dotenv()
env.garantir_diretorios()

print("ROOT_DIR:", ROOT_DIR)
print(env.resumo_ambiente())

## 2. Fonte canônica do Parquet

O artefato distribuído do projeto é identificado por três constantes em `src/utils/config.py`:

- `GITHUB_REPO` — repositório que hospeda a release;
- `DATA_VERSION` — tag da release;
- `PARQUET_NAME` — nome do arquivo dentro da release.

A URL efetiva é obtida por `C.srag_parquet_url()`, que também aceita
override via variável de ambiente `SRAG_PARQUET_URL` — útil para apontar
para uma release alternativa sem tocar no código.

In [ ]:
print(f"Repositório:  {C.GITHUB_REPO}")
print(f"Versão:       {C.DATA_VERSION}")
print(f"Arquivo:      {C.PARQUET_NAME}")
print(f"URL canônica: {C.srag_parquet_url()}")

## 3. Download do Parquet

`get_parquet()` delega a `src.dados` toda a parte de rede:
retry com backoff, verificação dos magic bytes `PAR1` e escrita do
sidecar `.sha256`.

Se o arquivo já existir localmente, ele é **reutilizado** — não baixa de novo.

In [ ]:
parquet_path = get_parquet()

print(f"\nParquet em: {parquet_path}")
print(f"Tamanho:    {parquet_path.stat().st_size / 1024**2:.2f} MB")

## 4. Leitura e verificação de sanidade

Carrega o Parquet em memória e confirma que o arquivo não está vazio.
As checagens mais profundas de schema e conteúdo ficam nos notebooks
de preparação da coorte.

In [ ]:
# ============================================================
# Verificação do Parquet sem carregar a base completa
# ============================================================

import time
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

inicio=time.perf_counter()
arquivo=pq.ParquetFile(parquet_path)
meta=arquivo.metadata
quantidade_registros=int(meta.num_rows)
quantidade_colunas=int(meta.num_columns)
quantidade_grupos_linhas=int(meta.num_row_groups)
tamanho_arquivo_mb=parquet_path.stat().st_size/1_048_576

if quantidade_registros<=0 or quantidade_colunas<=0:
    raise ValueError("Parquet vazio ou sem colunas.")

lote=next(arquivo.iter_batches(batch_size=5),None)
if lote is None:
    raise ValueError("Não foi possível extrair a amostra.")
amostra_parquet=lote.to_pandas(types_mapper=pd.ArrowDtype).reset_index(drop=True)
memoria_amostra_mb=amostra_parquet.memory_usage(index=True,deep=True).sum()/1_048_576

resumo_ingestao=pd.DataFrame({"metrica":["caminho","tamanho_mb","registros","colunas","row_groups","linhas_amostra","memoria_amostra_mb","tempo_s"],
"valor":[str(parquet_path),round(tamanho_arquivo_mb,2),quantidade_registros,quantidade_colunas,quantidade_grupos_linhas,len(amostra_parquet),round(memoria_amostra_mb,4),round(time.perf_counter()-inicio,3)]})
display(amostra_parquet)
display(resumo_ingestao)
print("Verificação concluída sem carregar o dataset completo.")


## 5. Diagnóstico rápido do Parquet

Leituras pequenas para conferir o acesso ao arquivo baixado do GitHub.

### 5.1 Diagnóstico rápido (rode isto antes de qualquer coisa)

In [ ]:
path = storage.get_parquet(force=False)

print("Arquivo:", path)
print("Existe:", path.exists())
print("Tamanho:", path.stat().st_size, "bytes")

with open(path, "rb") as f:
    print("Primeiros 20 bytes:", f.read(20))

In [ ]:
import pyarrow.parquet as pq
pq.read_schema(path)

### 5.2 Leitura mínima (só uma coluna)

Verifica se o Parquet é baixado/aberto corretamente.

In [ ]:
df = storage.read_srag(columns=["sg_uf"])
print("shape:", df.shape)
df.head()

### 5.3 Coluna categórica — municípios únicos

In [ ]:
storage.read_srag(columns=["id_municip"]).drop_duplicates().head(20)

### 5.4 Leitura com filtro (pushdown do Parquet)

Só as colunas necessárias e apenas linhas de SP.

In [ ]:
df_sp = storage.read_srag(
    columns=["dt_notific", "sg_uf", "evolucao"],
    filters=[("sg_uf", "==", "SP")],
)

print("shape:", df_sp.shape)
df_sp.head()